In [0]:
# Test 1 - Check secret scope is working
print(dbutils.secrets.get("practice-secret-scope", "client-id"))
print(dbutils.secrets.get("practice-secret-scope", "tenant-id"))
# Note: secret value won't print visibly, it will show [REDACTED] - that's normal

In [0]:
# Test 2 - Check storage connection
storage_account = "<storage-account>"  # replace this

spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", dbutils.secrets.get("practice-secret-scope", "client-id"))
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", dbutils.secrets.get("practice-secret-scope", "client-secret"))
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net", f"https://login.microsoftonline.com/{dbutils.secrets.get('practice-secret-scope', 'tenant-id')}/oauth2/token")

# Test listing the container
dbutils.fs.ls(f"abfss://unity-catalog-storage@{storage_account}.dfs.core.windows.net/")

In [0]:
# Test 2 - Check storage connection through Unity Catalog external location
# No Spark config needed - Unity Catalog handles authentication
storage_account = "<storage-account>"
#container = "unity-catalog-storage"
container = "practice-storage-container-bronze-data"

# Access through the external location (after creating it in Databricks UI)
result = dbutils.fs.ls(f"abfss://{container}@{storage_account}.dfs.core.windows.net/")
print(f"✓ Successfully accessed storage! Found {len(result)} items:")
for item in result[:10]:  # Show first 10 items
    print(f"  - {item.name}")

In [0]:
# Test writing a file to verify full access
storage_account = "<storage-account>"
container = "unity-catalog-storage"
test_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/test_file.txt"

try:
    # Write a test file
    dbutils.fs.put(test_path, "Hello from Databricks! This is a test file.", overwrite=True)
    print("✓ Successfully wrote test file")
    
    # Read it back
    content = dbutils.fs.head(test_path)
    print(f"✓ Successfully read test file: {content}")
    
    # List container again
    result = dbutils.fs.ls(f"abfss://{container}@{storage_account}.dfs.core.windows.net/")
    print(f"\n✓ Container now has {len(result)} item(s):")
    for item in result:
        print(f"  - {item.name} ({item.size} bytes)")
        
    print("\n🎉 All tests passed! Storage access is fully configured.")
except Exception as e:
    print(f"✗ Error: {type(e).__name__}")
    print(f"  {str(e)[:300]}")

In [0]:
# Test 3 - Check Unity Catalog external location
spark.sql("SHOW EXTERNAL LOCATIONS").show()

In [0]:
dbutils.fs.ls("abfss://raw-data@<storage-account>.dfs.core.windows.net/")